In [ ]:
import cv2
import os
import glob
import numpy as np
from wisardpkg import ClusWisard
import sys
sys.path.append('/mnt/c/Users/Isabella/tcc')
from wisard_object_tracker.src.utils import tracker_utils
import matplotlib.pyplot as plt

DATASET_ROOT_FOLDER = '/mnt/c/Users/Isabella/TCC/wisard_object_tracker/data/tiger2'
IMAGE_FOLDER = os.path.join(DATASET_ROOT_FOLDER, 'imgs')
GT_TXT_PATH = os.path.join(DATASET_ROOT_FOLDER, 'tiger2_gt.txt')

In [ ]:
# Carrega imagens
image_paths = sorted(glob.glob(os.path.join(IMAGE_FOLDER, '*.png')))
print(f"Encontradas {len(image_paths)} imagens")

# Carrega ground truths
all_ground_truths = tracker_utils.load_ground_truth_from_gt_txt(GT_TXT_PATH)

# Primeiro frame

In [ ]:
# Carrega primeiro frame  # lembrar de carregar certo em tons de cinza
first_frame = cv2.imread(image_paths[0])
print(f"Primeiro frame carregado: {first_frame.shape}")

first_gt = all_ground_truths[0]
print(f"Ground Truth do primeiro frame: {first_gt}")

# Mostra o frame com a bbox
first_frame_with_bbox = first_frame.copy()
x, y, w, h = map(int, first_gt)
cv2.rectangle(first_frame_with_bbox, (x, y), (x + w, y + h), (0, 255, 0), 2) 

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(first_frame_with_bbox, cv2.COLOR_BGR2RGB))
plt.title('Primeiro Frame - Objeto a Rastrear')
plt.axis('off')
plt.show()


In [ ]:
# --- PASSO 2: EXTRAIR E PRÉ-PROCESSAR PATCH ---
print("\n🔍 PASSO 2: Extraindo e pré-processando patch...")

# Extrai patch do objeto
patch = tracker_utils.extract_patch(first_frame, first_gt)

# Mostra o patch ORIGINAL (antes do pré-processamento)
plt.figure(figsize=(8, 6))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
plt.title(f'Patch Original\n shape:{patch.shape}')
plt.axis('off')


In [ ]:
# Pré-processa o patch
import cv2
import numpy as np

def preprocess_frame(frame):
    # Converte para grayscale caso seja RGB
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()

    # 1) MÉDIA GLOBAL  (correção do erro)
    global_mean = int(np.mean(gray))     # precisa ser escalar Python
    no_bg_global = cv2.subtract(gray, global_mean)

    # 2) Binarização Otsu
    _, otsu = cv2.threshold(
        no_bg_global, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # 3) Retornar vetor flatten binário
    vector = (otsu.flatten() > 0).astype(np.uint8)

    return vector

In [ ]:
# Pré-processa o patch
first_pattern = preprocess_frame(patch)
print(f"Primeiro Pattern pré-processado: {first_pattern.shape}")

In [ ]:
# --- PASSO 3: INSTANCIANDO CLUSWISARD ---

# Parâmetros simples
parameters = {
    "INPUT_PATTERN_SIDE": 32,                  
    "CLUSWISARD_ADDRESS_SIZE": 5,          
    "ACCEPTABLE_ACTIVATION_SCORE": 0.15,       
    "MIN_ACCEPTABLE_ACTIVATION_SCORE": 0.3,    
    "CLUSWISARD_MIN_SCORE": 0.4,               
    "CLUSWISARD_THRESHOLD": 4,                 
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 6,      
    "CLUSWISARD_BLEACHING_ACTIVATED": False,     
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "LAST_FRAME": 25,
    "MAX_SEARCH_WINDOW_SCALE": 1,
}

# Instancia ClusWisard

clus = ClusWisard(
            parameters['CLUSWISARD_ADDRESS_SIZE'], # address size
            parameters['CLUSWISARD_MIN_SCORE'], # min score
            parameters['CLUSWISARD_THRESHOLD'], # threshold
            parameters['CLUSWISARD_DISCRIMINATOR_LIMIT'], # discriminator limit
            bleachingActivated=parameters['CLUSWISARD_BLEACHING_ACTIVATED'],
            returnActivationDegree=parameters['CLUSWISARD_ACTIVATION_DEGREE'],
            returnConfidence=parameters['CLUSWISARD_RETURN_CONFIDENCE'],
            returnClassesDegrees=parameters['CLUSWISARD_CLASSES_DEGREES']
        )

In [ ]:
# Treina com o patch do objeto
clus.train([first_pattern.tolist()], ["object"])
print("✅ ClusWisard treinado com o patch do primeiro frame!")

In [ ]:
# --- PASSO 4: TESTAR SE RECONHECE O PRÓPRIO PATCH ---
print("\n✅ PASSO 4: Testando autorreconhecimento...")

# Classifica o MESMO patch usado no treinamento
result = clus.classify([first_pattern.tolist()])[0]
activation = result.get("activationDegree", -1)

print(f"Ativação com o próprio patch de treinamento: {activation}")
print(result)

# Região de Busca

In [ ]:
import numpy as np  # importa o NumPy para operações numéricas e trigonométricas

def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1, 
    step_size=0.5,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        
        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


### Gerando regiões para o primeiro frame

In [ ]:
# Parâmetros
search_window_scale = 0.5
step_size = 1          # quanto menor -> mais granular

search_gen = generate_search_regions_circular(
    prev_bbox = (16.0, 30.0, 34.0, 39.0),
    frame_shape = first_frame.shape,
    search_region_scale = search_window_scale,
    step_size = step_size
)

# Tracker

In [ ]:
import json

In [ ]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from wisardpkg import ClusWisard

# =========================
# PARÂMETROS (SEUS)
# =========================
parameters = {
    "INPUT_PATTERN_SIDE": 32,
    "CLUSWISARD_ADDRESS_SIZE": 7,
    "ACCEPTABLE_ACTIVATION_SCORE": 0.15,
    "MIN_ACCEPTABLE_ACTIVATION_SCORE": 0.3,
    "CLUSWISARD_MIN_SCORE": 0.4,
    "CLUSWISARD_THRESHOLD": 4,
    "CLUSWISARD_DISCRIMINATOR_LIMIT": 6,
    "CLUSWISARD_BLEACHING_ACTIVATED": False,
    "CLUSWISARD_ACTIVATION_DEGREE": True,
    "CLUSWISARD_RETURN_CONFIDENCE": True,
    "CLUSWISARD_CLASSES_DEGREES": True,
    "LAST_FRAME": 360,
    "MAX_SEARCH_WINDOW_SCALE": 1.3,
}

MAX_DISCRIMINATORS = parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"]
ACTIVATION_THRESHOLD = parameters["MIN_ACCEPTABLE_ACTIVATION_SCORE"]

print("\n🔄 Iniciando tracking sobre todos os frames...")

# =========================
# FUNÇÃO: CRIAR DISCRIMINADOR
# =========================
def create_discriminator():
    return ClusWisard(
        parameters["CLUSWISARD_ADDRESS_SIZE"],
        parameters["CLUSWISARD_MIN_SCORE"],
        parameters["CLUSWISARD_THRESHOLD"],
        parameters["CLUSWISARD_DISCRIMINATOR_LIMIT"],
        bleachingActivated=parameters["CLUSWISARD_BLEACHING_ACTIVATED"],
        returnActivationDegree=parameters["CLUSWISARD_ACTIVATION_DEGREE"],
        returnConfidence=parameters["CLUSWISARD_RETURN_CONFIDENCE"],
        returnClassesDegrees=parameters["CLUSWISARD_CLASSES_DEGREES"],
    )

# =========================
# INICIALIZAÇÃO (Algoritmo 1: passos 1–3)
# =========================
discriminator_queue = []

prev_bbox = first_gt
all_predictions = [prev_bbox]

# Primeiro frame
first_frame = cv2.imread(image_paths[0])
first_frame = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

first_patch = tracker_utils.extract_patch(first_frame, first_gt)
first_pattern = preprocess_frame(first_patch)

first_disc = create_discriminator()
first_disc.train([first_pattern.tolist()], ["object"])

# Insere no início da fila
discriminator_queue.insert(0, first_disc)

# =========================
# LOOP PRINCIPAL (Algoritmo 1: passos 4–14)
# =========================
for frame_idx, frame_path in enumerate(
    image_paths[1:parameters["LAST_FRAME"] + 1], start=1
):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} não carregado!")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    print(f"\n📷 Frame {frame_idx}: {frame_path}")

    # -------------------------
    # PASSO 5: REGIÕES DE BUSCA
    # -------------------------
    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=frame.shape,
        search_region_scale=parameters["MAX_SEARCH_WINDOW_SCALE"],
        step_size=1
    )

    all_regions = list(search_gen)
    print(f"🔍 Geradas {len(all_regions)} regiões")

    results = []

    # -------------------------
    # PASSO 5: CLASSIFICAÇÃO
    # TODOS OS DISCRIMINADORES
    # -------------------------
    for region in all_regions:

        patch = tracker_utils.extract_patch(frame, region)
        if patch.size == 0:
            continue

        pattern = preprocess_frame(patch)

        for disc_id, disc in enumerate(discriminator_queue):

            result = disc.classify([pattern.tolist()])[0]

            predicted_class = result.get("class", None)
            activation = result.get("activationDegree", -1)

            if predicted_class == "object":
                results.append({
                    "bbox": region,
                    "activation": activation,
                    "patch": pattern,
                    "disc_id": disc_id,
                    "confidence": result.get("confidence", None),
                })

    if not results:
        print("⚠️ Nenhuma região válida — mantendo bbox anterior.")
        all_predictions.append(prev_bbox)
        continue

    # -------------------------
    # PASSO 6: MELHOR REGIÃO
    # -------------------------
    best_region = max(results, key=lambda r: r["activation"])
    prev_bbox = best_region["bbox"]
    all_predictions.append(prev_bbox)

    print(
        f"🎯 Melhor bbox {best_region['bbox']} | "
        f"act={best_region['activation']:.4f} | "
        f"conf={best_region['confidence']} | "
        f"disc={best_region['disc_id']}"
    )

    # =========================
    # VISUALIZAÇÃO
    # =========================
    frame_vis = frame.copy()

    for r in results:
        x, y, w, h = map(int, r["bbox"])
        cv2.rectangle(frame_vis, (x, y), (x + w, y + h), (255, 0, 0), 1)

    bx, by, bw, bh = map(int, best_region["bbox"])
    cv2.rectangle(frame_vis, (bx, by), (bx + bw, by + bh), (0, 255, 0), 3)

    cv2.putText(
        frame_vis,
        f"BEST act={best_region['activation']:.2f} | D{best_region['disc_id']}",
        (bx, max(0, by - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

    best_patch = tracker_utils.extract_patch(frame, best_region["bbox"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(frame_vis)
    axes[0].set_title(f"Frame {frame_idx} - Regiões Testadas")
    axes[0].axis("off")

    axes[1].imshow(best_patch)
    axes[1].set_title(
        f"Melhor Patch\nact={best_region['activation']:.2f}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # -------------------------
    # PASSOS 7–13: RETREINO
    # -------------------------
    if 0.3 < best_region["activation"] < ACTIVATION_THRESHOLD + 0.3:

        print(
            f"⚡ Ativação baixa ({best_region['activation']:.4f}) "
            "→ Criando novo discriminador"
        )

        new_disc = create_discriminator()
        new_disc.train(
            [best_region["patch"].tolist()],
            ["object"]
        )

        # PASSO 12: insere no início
        discriminator_queue.insert(0, new_disc)

        # PASSOS 9–10: descarte FIFO
        if len(discriminator_queue) > MAX_DISCRIMINATORS:
            discriminator_queue.pop()

        print(f"🧠 Discriminadores ativos: {len(discriminator_queue)}")

# =========================
# FINAL
# =========================
print("\n✅ Tracking concluído!")
print(f"📦 Total final de discriminadores: {len(discriminator_queue)}")


In [ ]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt

print("\n🔄 Iniciando tracking sobre todos os frames...")

prev_bbox = first_gt
all_predictions = [prev_bbox]

# --- ACUMULADORES DE RETREINO ---
retrain_patches = []
retrain_info = []

for frame_idx, frame_path in enumerate(image_paths[1:364], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} ({frame_path}) não carregado!")
        continue

    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    print(f"\n📷 Frame {frame_idx}: {frame_path}")
    if frame_idx <100:
        scale=1
    else: 
        scale=1

    # --- PASSO 1: Gerar regiões de busca ---
    search_gen = generate_search_regions_circular(
        prev_bbox=prev_bbox,
        frame_shape=frame.shape,
        search_region_scale=scale,
        step_size=1
    )

    all_regions = list(search_gen)
    print(f"🔍 Geradas {len(all_regions)} regiões")

    # --- PASSO 2: Classificar regiões ---
    results = []

    for i, region in enumerate(all_regions):

        patch_region = tracker_utils.extract_patch(frame, region)
        if patch_region.size == 0:
            continue

        pattern_region = preprocess_frame(patch_region)

        result = clus.classify([pattern_region.tolist()])[0]
        predicted_class = result.get("class", None)
        activation = result.get("activationDegree", -1)
        
        # ✅ Só considera se a classe for "object"
        if predicted_class == "object":
            results.append({
                "region_id": i,
                "bbox": region,
                "activation": activation,
                "patch": pattern_region,
                "class": predicted_class,
                "confidence": result.get("confidence", None),
            })

    if not results:
        print("⚠️ Nenhuma região válida — mantendo bbox anterior.")
        all_predictions.append(prev_bbox)
        continue

    # --- PASSO 3: Escolher melhor região ---
    best_region = max(results, key=lambda r: r["activation"])
    prev_bbox = best_region["bbox"]
    all_predictions.append(prev_bbox)

    print(
        f"🎯 Melhor região: {best_region['bbox']} "
        f"| ativação={best_region['activation']:.4f}"
    )

    # --- FRAME COM TODAS AS REGIÕES DESENHADAS ---
    frame_vis = frame.copy()

    # 🔴 Todas as regiões testadas
    for r in results:
        x, y, w, h = map(int, r["bbox"])
        cv2.rectangle(
            frame_vis,
            (x, y),
            (x + w, y + h),
            (255, 0, 0),
            1
        )

    # 🟢 Melhor região
    bx, by, bw, bh = map(int, best_region["bbox"])
    cv2.rectangle(
        frame_vis,
        (bx, by),
        (bx + bw, by + bh),
        (0, 255, 0),
        3
    )

    cv2.putText(
        frame_vis,
        f"BEST act={best_region['activation']:.2f}",
        (bx, max(0, by - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2
    )

    # --- MELHOR PATCH ---
    best_patch = tracker_utils.extract_patch(frame, best_region["bbox"])

    # --- VISUALIZAÇÃO LADO A LADO ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].imshow(frame_vis)
    axes[0].set_title(f"Frame {frame_idx} - Regiões Testadas")
    axes[0].axis("off")

    axes[1].imshow(best_patch)
    axes[1].set_title(
        f"Melhor Patch\nact={best_region['activation']:.2f}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    # --- RETREINO CONDICIONAL ---
    if best_region["activation"] < 0.4:
        print(
            f"⚡ Ativação baixa ({best_region['activation']:.4f}) "
            "→ Retreinando ClusWisard"
        )

        clus.train([best_region["patch"].tolist()], ["object"])

        retrain_patches.append(best_patch)
        retrain_info.append({
            "frame": frame_idx,
            "activation": best_region["activation"],
            "bbox": best_region["bbox"],
        })

        clus_dict = json.loads(clus.json())
        clusters = clus_dict["clusters"]

        discriminadores_por_cluster = {
            nome: len(c["discriminators"])
            for nome, c in clusters.items()
        }

        print("🧠 Discriminadores:", discriminadores_por_cluster)

# --- FINAL ---
print("\n✅ Tracking concluído!")

# --- VISUALIZAR PATCHES DE RETREINO ---
print("\n🧠 Patches utilizados para retreino:")
print(f"Total: {len(retrain_patches)}")

for i, (patch, info) in enumerate(zip(retrain_patches, retrain_info)):

    print(
        f"Patch {i} | Frame {info['frame']} | "
        f"Ativação {info['activation']:.4f} | "
        f"BBox {info['bbox']}"
    )

    plt.figure(figsize=(4, 4))
    plt.title(
        f"Retreino {i} - Frame {info['frame']} "
        f"(act={info['activation']:.2f})"
    )
    plt.imshow(patch)
    plt.axis("off")
    plt.show()


In [ ]:
import cv2
import os
import shutil


# --- Pasta de saída ---
output_dir = "resultados/"

# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Cria nova pasta vazia
os.makedirs(output_dir, exist_ok=True)
# --- Salvar cada frame com a bbox prevista ---
for frame_idx, frame_path in enumerate(image_paths[1:]):
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} não carregado!")
        continue

    # Pega a previsão correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(frame, f"Frame {frame_idx}", (x, max(20, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Salva imagem
    output_path = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")
    cv2.imwrite(output_path, frame)

print(f"✅ {len(all_predictions)} frames salvos em: {output_dir}")


In [ ]:
import cv2
import os
import shutil

# --- Pasta de saída ---
output_dir = "resultados_video/"

# Apaga a pasta se existir
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

# --- Configuração do vídeo ---
video_path = "resultados_video/tracking_output.mp4"
fps = 1  # ajuste se seu vídeo original tiver outro FPS

# Lê o primeiro frame para pegar o tamanho
first_frame = cv2.imread(image_paths[1])
h, w = first_frame.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
video_writer = cv2.VideoWriter(video_path, fourcc, fps, (w, h))

# --- Processar e gravar vídeo ---
for frame_idx, frame_path in enumerate(image_paths[1:364], start=1):

    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"❌ Frame {frame_idx} não carregado!")
        continue

    # Pega a bbox prevista correspondente
    if frame_idx < len(all_predictions):
        x, y, bw, bh = map(int, all_predictions[frame_idx])
        cv2.rectangle(frame, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
        cv2.putText(
            frame,
            f"Frame {frame_idx}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
        )

    # Escreve no vídeo
    video_writer.write(frame)

video_writer.release()
print(f"🎬 Vídeo salvo com sucesso em: {video_path}")
